# Fixed-window pool-50: reranker depth và đầu vào LightGBM

Notebook hậu xử lý cho output của `kaggle_evaluate_fixed448_pool50_3epochs.ipynb`. Chỉ đọc archive đã lưu; không chạy lại retrieval, cross-encoder hay LightGBM.

Cross-encoder chấm tối đa 50 chunk rồi gộp điểm về `aid`. LightGBM nhận **toàn bộ danh sách aid đã rerank** (tối đa 50 aid/query, thường ít hơn vì nhiều chunk có thể cùng một aid). `lightgbm_top_2` là số aid LightGBM trả ra cuối cùng, không phải độ sâu đầu vào. Notebook xác minh feature rows đã lưu có đúng bằng toàn bộ danh sách reranker không.

In [ ]:
from pathlib import Path
import json, tarfile
import numpy as np
import pandas as pd
from IPython.display import display

INPUT_ROOT = Path('/kaggle/input')
WORK = Path('/kaggle/working/fixed448_pool50_3epoch_depth_summary')
WORK.mkdir(parents=True, exist_ok=True)
ARCHIVE_NAME = 'fixed448_pool50_refoff_3epoch_checkpoint_eval_results.tar.gz'
archives = list(INPUT_ROOT.rglob(ARCHIVE_NAME))
if len(archives) != 1:
    raise RuntimeError(f'Attach exactly one output archive from the 3-epoch checkpoint evaluation; found: {archives}')

EXTRACTED = WORK / 'source'
if not (EXTRACTED / 'checkpoint_scoreboard.csv').is_file():
    EXTRACTED.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archives[0], 'r:gz') as bundle:
        bundle.extractall(EXTRACTED, filter='data')
roots = [p.parent for p in EXTRACTED.rglob('checkpoint_scoreboard.csv') if (p.parent / 'checkpoint_runs').is_dir()]
if len(roots) != 1:
    raise RuntimeError(f'Could not identify one complete evaluation output root: {roots}')
RESULT_ROOT = roots[0]
print({'archive': str(archives[0]), 'result_root': str(RESULT_ROOT)})

## 1. Kiểm tra candidate list mà LightGBM nhận

Feature archive `enhanced_v2_dev_feature_rows.npz` lưu một dòng cho mỗi cặp query/article sau rerank. So sánh `(qid, aid, original_rank)` với `predictions.jsonl` để phát hiện trường hợp LightGBM chỉ nhận một prefix hoặc artifact bị lệch.

In [ ]:
CHECKPOINTS = {
    'Chưa fine-tune': 'warm_start_vietnamese',
    'Epoch 1': 'adaptation_epoch_1',
    'Epoch 2': 'adaptation_epoch_2',
    'Epoch 3': 'adaptation_epoch_3',
}
def read_jsonl(path):
    with path.open(encoding='utf-8') as stream:
        return [json.loads(line) for line in stream if line.strip()]

candidate_rows = []
reference_candidates = None
for display_name, label in CHECKPOINTS.items():
    run_dir = RESULT_ROOT / 'checkpoint_runs' / f'fixed448_pool50_{label}'
    prediction_path = run_dir / 'predictions.jsonl'
    feature_path = RESULT_ROOT / 'lightgbm_dev' / label / 'enhanced_v2_dev_feature_rows.npz'
    for required in (prediction_path, feature_path, run_dir / 'metrics_by_output_k.json'):
        if not required.is_file():
            raise FileNotFoundError(f'Missing saved stage artifact: {required}')
    predictions = read_jsonl(prediction_path)
    by_qid = {int(row['qid']): [int(aid) for aid in row['retrieved_aids']] for row in predictions}
    if len(by_qid) != len(predictions) or len(by_qid) != 438:
        raise RuntimeError(f'{label}: expected 438 unique dev queries, found {len(by_qid)}')
    if any(len(aids) != len(set(aids)) for aids in by_qid.values()):
        raise RuntimeError(f'{label}: duplicate aid in reranker ranking')
    if max(map(len, by_qid.values())) > 50:
        raise RuntimeError(f'{label}: found more than 50 reranked aids for a query')
    signatures = {qid: frozenset(aids) for qid, aids in by_qid.items()}
    if reference_candidates is None:
        reference_candidates = signatures
    elif signatures != reference_candidates:
        raise RuntimeError(f'{label}: candidate aid set differs from other checkpoints')
    with np.load(feature_path, allow_pickle=False) as features:
        feature_rows = {(int(q), int(a), int(r)) for q, a, r in zip(features['qids'], features['aids'], features['original_ranks'])}
    expected_rows = {(qid, aid, rank) for qid, aids in by_qid.items() for rank, aid in enumerate(aids, start=1)}
    if feature_rows != expected_rows:
        raise RuntimeError(f'{label}: LightGBM feature rows do not cover exactly the full reranker ranking; expected={len(expected_rows)}, found={len(feature_rows)}')
    counts = np.asarray([len(aids) for aids in by_qid.values()], dtype=int)
    candidate_rows.append({'model': display_name, 'queries': len(counts), 'lightgbm_input_rows': int(counts.sum()), 'mean_aids_per_query': counts.mean(), 'median_aids_per_query': float(np.median(counts)), 'min_aids_per_query': int(counts.min()), 'max_aids_per_query': int(counts.max()), 'queries_with_50_aids': int((counts == 50).sum()), 'input_verified_as_full_reranker_list': True})

candidate_audit = pd.DataFrame(candidate_rows)
display(candidate_audit.round(3))
candidate_audit.to_csv(WORK / 'lightgbm_input_depth_audit.csv', index=False)

## 2. So sánh reranker tại các cutoff và sau LightGBM

`cutoff=50` bao phủ toàn bộ danh sách aid (mỗi query có không quá 50 aid), tức là trả cả input pool ra; riêng con số này không đo thứ tự ranking. Các cutoff nhỏ hơn cho biết chất lượng phần đầu ranking. Bảng LightGBM dùng cùng full input list; policy chỉ mô tả cách chọn đầu ra.

In [ ]:
reranker_rows = []
cutoffs = ['2', '5', '10', '20', '30', '50']
for display_name, label in CHECKPOINTS.items():
    run_dir = RESULT_ROOT / 'checkpoint_runs' / f'fixed448_pool50_{label}'
    fixed = json.loads((run_dir / 'metrics_by_output_k.json').read_text(encoding='utf-8'))['results']
    missing = [k for k in cutoffs if k not in fixed]
    if missing:
        raise RuntimeError(f'{label}: missing reranker cutoffs {missing}; found {list(fixed)}')
    for cutoff in cutoffs:
        metric = fixed[cutoff]
        reranker_rows.append({'model': display_name, 'stage': 'Cross-encoder only', 'candidate_depth': 'full list from 50 chunks (<=50 aids)', 'output_policy': f'fixed top-{cutoff}' if cutoff != '50' else 'all candidates (cutoff 50)', 'precision': metric['precision'], 'recall': metric['recall'], 'f2': metric['f2']})

reranker_table = pd.DataFrame(reranker_rows)
display(reranker_table.round(4))
reranker_table.to_csv(WORK / 'reranker_cutoff_metrics.csv', index=False)

lgbm_board = pd.read_csv(RESULT_ROOT / 'lightgbm_checkpoint_scoreboard.csv')
lgbm_policies = {
    'lightgbm_top_2': 'fixed top-2 output',
    'lightgbm_dev_threshold_fallback_2': 'dev threshold + fallback 2',
    'lightgbm_nested_oof_score_gap': 'nested OOF score-gap',
    'lightgbm_pooled_oof_score_gap_dev_tuned': 'pooled OOF score-gap (dev-tuned)',
}
lgbm_rows = lgbm_board[lgbm_board['policy'].isin(lgbm_policies)].copy()
lgbm_rows['model'] = lgbm_rows['model'].map({label: name for name, label in CHECKPOINTS.items()})
lgbm_rows['input_depth'] = 'all reranked aids (verified; <=50/query)'
lgbm_rows['output_policy'] = lgbm_rows['policy'].map(lgbm_policies)
lgbm_rows = lgbm_rows[['model', 'input_depth', 'output_policy', 'precision', 'recall', 'f2']].sort_values(['output_policy', 'f2'], ascending=[True, False])
display(lgbm_rows.round(4))
lgbm_rows.to_csv(WORK / 'lightgbm_policy_metrics.csv', index=False)
print({'outputs': [str(WORK / name) for name in ('lightgbm_input_depth_audit.csv', 'reranker_cutoff_metrics.csv', 'lightgbm_policy_metrics.csv')]})

## Cách dùng kết quả

- Dùng `reranker_cutoff_metrics.csv` để trình bày reranker độc lập; các cutoff nhỏ đo chất lượng phần đầu ranking, còn `all candidates (cutoff 50)` cho biết kết quả nếu trả toàn bộ input pool.
- Dùng `lightgbm_policy_metrics.csv` để trình bày kết quả sau LightGBM. `top-2` là số aid đầu ra sau khi LightGBM đã chấm toàn bộ candidate list.
- Nếu chọn LightGBM input-depth 30 thay vì toàn bộ candidate list, phải dùng kết quả depth-30 đã fit/evaluate riêng; không thể gọi top-30 output là depth-30 input. Threshold và pooled OOF là dev-tuned; nested OOF là đánh giá fold-safe hơn.